# Demo — Governing a Healthcare Medallion Lakehouse
**Veridian Health** operates a multi-jurisdiction digital telehealth and diagnostics platform across four clinical & operational domains (**Patient Identity & Intake**, **Clinical Telehealth**, **Diagnostics & Labs**, and **Revenue Cycle & Quality Reporting**) organized into a **Bronze / Silver / Gold Medallion Lakehouse** on AWS.

Before choosing a governance operating model (centralized, decentralized, or federated) and building a RACI matrix, inspect the lakehouse catalog and cross-domain semantic conflicts under **HIPAA (45 CFR §164)**, **GDPR Art. 9 (Special Category Health Data)**, and **CMS eCQM (Electronic Clinical Quality Measures)**.

## Step 0: Connect to your Cloud Lab

Run the cell below to configure your AWS Cloud Lab environment and seed the **Veridian Health Medallion Lakehouse** (`veridian_lakehouse` in AWS Glue, S3 `bronze/silver/gold` Parquet prefixes, and AWS Lake Formation enforcement) via `setup_aws.py`.

In [1]:
import os
os.environ["VERIDIAN_LF_LIVE"] = "1"
os.environ["VERIDIAN_AWS_REGION"] = "us-east-1"
# If your Vocareum AWS credentials are not already in the environment, paste them below:
# os.environ["AWS_ACCESS_KEY_ID"] = "..."
# os.environ["AWS_SECRET_ACCESS_KEY"] = "..."
# os.environ["AWS_SESSION_TOKEN"] = "..."

import boto3
import setup_aws
glue = boto3.client("glue", region_name=os.environ["VERIDIAN_AWS_REGION"])
try:
    glue.get_database(Name="veridian_lakehouse")
    print("Glue database 'veridian_lakehouse' is already seeded and ready.")
except glue.exceptions.EntityNotFoundException:
    setup_aws.main()

Glue database 'veridian_lakehouse' is already seeded and ready.


In [2]:
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from demo_toolkit import connect_lakehouse, lh_backend

lh = connect_lakehouse()
print(f"Backend: {lh_backend()} -> {type(lh).__name__} (Glue DB: {lh.database}, Region: {lh.region})")
display(pd.DataFrame([
    {"table": t, "zone": tm.zone, "domain": tm.domain, "steward": tm.steward,
     "sensitivity": tm.sensitivity, "retention_policy": tm.retention_policy}
    for t, tm in lh.catalog.tables.items()
]))

Backend: live -> AwsMedallionLakehouse (Glue DB: veridian_lakehouse, Region: us-east-1)

                           table    zone                             domain                           steward   sensitivity retention_policy
0          silver_patient_master  silver          Patient Identity & Intake  patient-identity@veridian.health           PHI              P6Y
1          bronze_patient_intake  bronze          Patient Identity & Intake  patient-identity@veridian.health           PHI              NaN
2      silver_clinical_encounter  silver                Clinical Telehealth                               NaN  CONFIDENTIAL              P6Y
3              silver_lab_result  silver                 Diagnostics & Labs   diagnostics-lab@veridian.health           NaN              P7Y
4  bronze_lab_device_calibration  bronze                 Diagnostics & Labs   diagnostics-lab@veridian.health  CONFIDENTIAL              NaN
5           silver_claim_billing  silver  Revenue Cycle & Quality

## Why pure decentralization fails under HIPAA, GDPR Art. 9 & CMS eCQM

In a purely decentralized model, each clinical domain defines its own patient keys, research consent flags, and lab units across Bronze and Silver zones. Inspect the three cross-domain conflicts surfaced in `lh.semantic_conflicts()`.

In [3]:
print("--- Cross-domain semantic conflicts in Veridian Health ---")
display(lh.semantic_conflicts())

--- Cross-domain semantic conflicts in Veridian Health ---

                                 concept                   domain_a                                                      field_a                           domain_b                                                field_b                                                                                                         impact
0                   patient identity key  Patient Identity & Intake  silver_patient_master.mpi_id (INTEGER Master Patient Index)  Revenue Cycle & Quality Reporting  silver_claim_billing.patient_email_ref (email string)                    No shared surrogate key; cross-domain clinical/billing joins rely on mutable email strings.
1   secondary-use research authorization  Patient Identity & Intake                    silver_patient_master.hipaa_research_auth                Clinical Telehealth      silver_clinical_encounter.session_research_opt_in  Session opt-in disagrees with HIPAA authorization SoR ~18% o

## How to structure a RACI row (the single-Accountable rule)

In a **federated Medallion operating model**, central Health Information Management (HIM) governance sets platform-wide standards (PHI classification taxonomy, HIPAA research-authorization system of record, and Medallion promotion gates), while clinical domain stewards execute day-to-day stewardship on their Bronze and Silver tables.

Every governance function must have **exactly one Accountable (`A`) role**:

| Governance Function | Chief Health Data Officer (Central) | Patient Identity Steward | Clinical Telehealth Steward | Diagnostics Lab Steward | Revenue & Quality Steward | Data Platform Eng |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: |
| **PHI / HIPAA classification standard** | **A** | R | R | R | R | C |
| **HIPAA research authorization SoR** | **A** | R (MPI SoR) | C (consumes flag) | I | I | I |

## Sketching the federated Medallion topology in Mermaid

```mermaid
flowchart TB
    CHDO["Central Health Governance Core\n(CHDO + CMS eCQM Lead + Data Platform Eng)\nPHI taxonomy • HIPAA auth SoR • Medallion SLO gates"]
    D1["Patient Identity & Intake\nbronze_patient_intake -> silver_patient_master"]
    D2["Clinical Telehealth\nsilver_clinical_encounter"]
    D3["Diagnostics & Labs\nbronze_lab_device_calibration -> silver_lab_result"]
    D4["Revenue Cycle & Quality\nsilver_claim_billing -> gold_cms_timely_care"]

    CHDO -- "PHI classification & Lake Formation policies" --> D1 & D2 & D3 & D4
    D1 -- "authoritative hipaa_research_auth (MPI SoR)" --> CHDO
    CHDO -- "enforces research opt-in sync" --> D2
    D2 & D3 -- "minutes-normalized lab turnaround" --> D4
```